# Wealth-distribution calibration on the Colab T4

Calibrates heterogeneous capital productivity against the STATIONARY WEALTH
DISTRIBUTION rather than a return curve directly (the sibling
`colab_ks_kappa_calibration.ipynb` experiment's design, reworked to fix its
input/output conflation -- see `runs/ks-heterogeneous-wealth/README.md`).

Xavier (2021)'s digitized return-by-percentile bars are interpolated into a
smooth quantile function, agent rank is mapped through it via a
**Beta-distorted** uniform grid (`(a,b)=(1,1)` = no distortion = literally
Xavier's own curve), and three free parameters -- `scale` (mean kappa),
`a`, `b` (Beta shape) -- are placed by the same Bayesian-optimization
machinery as the sibling experiment (`scikit-optimize`'s `gp_minimize`,
natively extended to 3 dimensions) to match `targets`: default is the
top-10%-wealth-share figure (0.70) already cited in the paper's own text.

Default config: `n_agents=1000`, `num_envs=8`, `bo_calls=50` (8 initial design
+ 42 GP-guided -- more than the sibling's 13, since 3-D needs more points).
**Time the first evaluation before assuming the rest fit in one Colab
session.** Every evaluation prints the actual envs/agents/memory footprint,
and results.csv + the current-best raw rollout/params are checkpointed after
every evaluation so a disconnect partway through loses only the run in
progress.


In [ ]:
# Setup: clone or update the repo, install (idempotent -- safe to re-run).
%cd /content
![ -d jax-marl-bc ] || git clone https://github.com/danmonuni/jax-marl-bc.git
%cd jax-marl-bc
!git pull
!pip install -q -r requirements.txt && pip install -q -e . --no-deps


In [ ]:
# Sanity: a GPU runtime is attached (Runtime > Change runtime type > T4 GPU).
!nvidia-smi -L


In [ ]:
# Mount Drive BEFORE the run so the result is saved as soon as it finishes
# (a Colab disconnect then loses at most the run in progress, never a
# finished one).
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

def save_results(name='ks-heterogeneous-wealth'):
    """Sync runs/<name>/results -> Drive (exact path, idempotent re-sync)."""
    src = f'runs/{name}/results'
    dst = f'/content/drive/MyDrive/jax-marl-bc-runs/{name}/results'
    assert os.path.exists(src), f"{src} missing - did the calibration run finish?"
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"saved {src} -> {dst}")


## Calibration search

Runs `runs/ks-heterogeneous-wealth/config.yaml` as-is: `n_agents=1000`,
`device=gpu`, log-uniform search over `scale_bounds=[0.3,3.0]`,
`a_bounds`/`b_bounds=[0.1,10.0]`, targeting `top_0.1_share=0.70`. Pass
dotlist overrides after the script path to change any of these -- e.g. fewer
calls for a first timing check, or adding a second target:
`!python runs/ks-heterogeneous-wealth/calibrate_wealth_distortion.py bo_calls=6 bo_init_points=3`
`!python runs/ks-heterogeneous-wealth/calibrate_wealth_distortion.py "targets={top_0.1_share: 0.70, capital_gini: 0.85}"`


In [ ]:
!python runs/ks-heterogeneous-wealth/calibrate_wealth_distortion.py


In [ ]:
save_results('ks-heterogeneous-wealth')


## Results


In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('runs/ks-heterogeneous-wealth/results/results.csv')
display(df[['scale', 'a', 'b', 'score', 'capital_gini', 'top_0.1_share', 'top_0.01_share']])

fig_path = 'runs/ks-heterogeneous-wealth/results/calibration_fit.png'
display(Image(fig_path))
